In [ ]:
import ee
# import geemap.core as geemap  # in colab
import geemap
import eerepr 

In [ ]:
# ee.Authenticate()

# ee.Initialize()

# Data

## e-Bird points

In [ ]:
pts = ee.FeatureCollection("users/kirara/PhD_Bonn/Project_tradeoff_grassland/eBird/richness_2018") 

In [ ]:
pts

In [ ]:
pts.first()

In [ ]:
print(pts.size().getInfo())

## Export tiles

In [ ]:
borderTiles_256 = ee.FeatureCollection('users/hadicu06/Postdoc_Bonn/exportTiles/2024_02_14_tiles256_intersectCountryBorders101Km_4cac757c209aa8f0d7458d74349d777f');

borderTiles_1024 = ee.FeatureCollection('users/hadicu06/Postdoc_Bonn/exportTiles/2024_02_14_tiles1024_intersectCountryBorders101Km_4cac757c209aa8f0d7458d74349d777f');

# Covariates

In [ ]:
# In the exportData() function

# Analysis parameters

In [ ]:
params = {}

### Choose export tiles

In [ ]:
params['exportTiles'] = borderTiles_1024

### Choose output spatial resolution (data extraction footprint)

In [ ]:
params['outputScale'] = 1000  # 1000 meters

###  Define start and end dates

In [ ]:
params['startdate'] = '2018-01-01'
params['enddate'] = '2018-12-31'

In [ ]:
params

# Export data

## Define the export data function

In [ ]:
def exportDataByTileAllLayersFunc(    
    exportTiles,
    outputScale,             
    driveFolder,
    parallelizationDesign,
    outputAssetName,
    tileScale=1
):    
       
    # ------- Loop over tiles --------

    tileIds = exportTiles.distinct('tileID').aggregate_array('tileID').getInfo()   

    counter = 0


    for tileId in tileIds:   # for each region
   
        counter += 1
        
        print("counter number ", counter)
        
        print("processing tileId ", tileId)
            
        # ------- Processing bounds for the given tile --------

        tile = exportTiles.filter(ee.Filter.eq("tileID", tileId)).first()

        geom = tile.geometry()

        geomSimple = geom   # already simple cause a tile

        geomSimpleBuff = geomSimple.buffer(params['outputScale']*1.5)  # just to be safe if pixel value needs to be interpolated from surrounding adjacent pixels...


        # ------- Filter the points to the tile bound --------

        ptsFiltered = pts.filterBounds(geomSimple)

        
        # ------- Continue data extraction and export only if tile is not empty of the extraction points --------

        if not ptsFiltered.size().getInfo() == 0:                         # there can be border tile with no greenness points!!       
          
            
            # ------- Covariates --------


            # Define start and end dates
            startdate = params['startdate']
            enddate = params['enddate']

            # Load TerraClimate dataset
            terraClimate = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE') \
                .filterDate(startdate, enddate) \
                .filterBounds(geomSimpleBuff)   # filter bound

            # Calculate precipitation sum
            precipitation = terraClimate.select('pr') \
                .filterDate(startdate, enddate) \
                .reduce(ee.Reducer.sum()) \
                .clip(geomSimpleBuff) 

            # Calculate mean minimum temperatu
            temperaturemin = terraClimate.select('tmmn') \
                .filterDate(startdate, enddate) \
                .reduce(ee.Reducer.mean()) \
                .clip(geomSimpleBuff) 

            # Calculate mean maximum temperature
            temperaturemax = terraClimate.select('tmmx') \
                .filterDate(startdate, enddate) \
                .reduce(ee.Reducer.mean()) \
                .clip(geomSimpleBuff) 

            # Calculate mean solar radiation
            srad = terraClimate.select('srad') \
                .filterDate(startdate, enddate) \
                .reduce(ee.Reducer.mean()) \
                .clip(geomSimpleBuff) 

            # Load GMTED dataset
            gmted = ee.Image('USGS/GMTED2010') \
                 .clip(geomSimpleBuff) 
            # Extract elevation
            elevation = gmted.select('be75').rename('gmted_elevation') 
            # Calculate terrain slope
            gmted_terrain = ee.Algorithms.Terrain(elevation)
            slope = gmted_terrain.select('slope').rename('gmted_slope')


            # Load MODIS EVI dataset and apply QA mask
            def applyQA_mask_modis_evi(image):
                evi = image.select('EVI')
                qa = image.select('SummaryQA')
                qa_mask = qa.eq(0)
                maskedEVI = evi.updateMask(qa_mask)
                return maskedEVI

            evi = ee.ImageCollection('MODIS/061/MOD13A3') \
                .filterDate(startdate, enddate) \
                .filterBounds(geomSimpleBuff) \
                .select(['EVI', 'SummaryQA']) \
                .map(applyQA_mask_modis_evi) \
                .select('EVI') \
                .reduce(ee.Reducer.mean()) \
                .clip(geomSimpleBuff) 

            # Load MODIS NPP dataset
            npp = ee.ImageCollection("MODIS/061/MOD17A3HGF") \
                .filterDate(startdate, enddate) \
                .filterBounds(geomSimpleBuff) \
                .select('Npp') \
                .first() \
                .clip(geomSimpleBuff) 

            # Load land cover datasets
            natural_g = ee.Image("users/kirara/PhD_Bonn/Project_tradeoff_grassland/landcover/land_cover_2012") \
                .select('b4') \
                .clip(geomSimpleBuff) 

            managed_g = ee.Image("users/kirara/PhD_Bonn/Project_tradeoff_grassland/landcover/land_cover_2012") \
                .select('b6') \
                .clip(geomSimpleBuff) 

            wild_g = ee.Image("users/kirara/PhD_Bonn/Project_tradeoff_grassland/landcover/land_cover_2012") \
                .select('b5') \
                .clip(geomSimpleBuff) 

            forest = ee.Image("users/kirara/PhD_Bonn/Project_tradeoff_grassland/landcover/land_cover_2012") \
                .select('b2') \
                .clip(geomSimpleBuff) 

            crop = ee.Image("users/kirara/PhD_Bonn/Project_tradeoff_grassland/landcover/land_cover_2012") \
                .select('b3') \
                .clip(geomSimpleBuff) 

            # Calculate soil moisture
            moisture = ee.ImageCollection("NASA/GLDAS/V021/NOAH/G025/T3H") \
                .filterDate(startdate, enddate) \
                .filterBounds(geomSimpleBuff) \
                .select('SoilMoi0_10cm_inst') \
                .reduce(ee.Reducer.sum()) \
                .clip(geomSimpleBuff) 

            # Calculate mean wind speed
            wind = ee.ImageCollection("NASA/GLDAS/V021/NOAH/G025/T3H") \
                .filterDate(startdate, enddate) \
                .filterBounds(geomSimpleBuff) \
                .select('Wind_f_inst') \
                .reduce(ee.Reducer.mean()) \
                .clip(geomSimpleBuff) 

            # Stack the covariates
            covariates = ee.Image.cat([
                precipitation.rename('precipitation'),
                temperaturemin.rename('tmin'),
                temperaturemax.rename('tmax'),
                elevation.rename('elevation'),
                slope.rename('slope'),
                srad.rename('srad'),
                evi.rename('evi'),
                npp.rename('npp'),
                natural_g.rename('natural_g'),
                managed_g.rename('managed_g'),
                wild_g.rename('wild_g'),
                forest.rename('forest'),
                crop.rename('cropland'),
                moisture.rename('moisture'),
                wind.rename('wind')
            ])



            # ------- Extract rasters values at points --------
                
            # Three ways:
            # (1) pts.map(extractAPoint)
            # (3) reduceRegions                


            if parallelizationDesign=='1':

                ####### Way (1) pts.map(extractAPoint) #################

                def extractAPoint(point):
                    output = (ee.Feature(point.geometry(),
                                        covariates
                                        .reduceRegion(                                               
                                            reducer=ee.Reducer.mean(),  ## assumes all covariates are continuous and can be spatially aggregated by mean
                                            geometry=point.geometry(),
                                            scale=outputScale,
                                            maxPixels=1e13, 
                                            tileScale=tileScale
                                        ))).copyProperties(point)
                    return output   
                
                sampled = ptsFiltered.map(extractAPoint)

           

            elif parallelizationDesign=='2':  

                ####### Way (2) reduceRegions #################       

                sampled = (covariates
                    .reduceRegions(
                        collection=ptsFiltered, 
                        reducer=ee.Reducer.mean(), ## assumes all covariates are continuous and can be spatially aggregated by mean
                        scale=outputScale,
                        tileScale=tileScale 
                    ))

            # -------Add lat long  --------

            def addLatLonFunc(feature):
                return feature.set({'latitude': feature.geometry().coordinates().get(1),
                                    'longitude': feature.geometry().coordinates().get(0)})

            sampled = sampled.map(addLatLonFunc)


            # print(sampled.limit(2).getInfo())         ## debug
            # return sampled.limit(2)                   ## debug


            # ------- Export to Drive  --------

            outputAssetName = outputAssetName + '_' + str(tileId)                                   ## change!!! 

            task = ee.batch.Export.table.toDrive(
                    collection=sampled,
                    description=outputAssetName,
                    folder=driveFolder,
                    fileFormat='CSV',
                )

            task.start()

            print(outputAssetName, "started")     

## Run the export data function

### Test for a few tiles

In [ ]:
params['exportTiles'].limit(2)

In [ ]:
not_run = False    # to do -> doing -> done OK

if not not_run:
    exportDataByTileAllLayersFunc(    
        exportTiles=params['exportTiles'].limit(2),  ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE_v2',
        parallelizationDesign='2',
        outputAssetName='2024-05-06_eBird_2018_withCovariates_test',
        tileScale=1
)



### Now run for all tiles

In [ ]:
not_run = True    # to do

if not not_run:
    exportDataByTileAllLayersFunc(    
        exportTiles=params['exportTiles'].limit(2),  ##
        outputScale=params['outputScale'],             
        driveFolder='POSTDOC_BONN_GEE_v2',
        parallelizationDesign='2',
        outputAssetName='2024-05-06_eBird_2018_withCovariates_test',
        tileScale=1
)
